# 01 — Load and prepare the data

**Stage:** obtaining and preparing the data.

This notebook:
1. reads the original UCI file `data/raw/bank-additional-full.csv`;
2. checks it (size, target, missing values);
3. rebuilds the **year** of each call, which the file does not contain;
4. assigns each call to a time period (train, validation, test 1, test 2);
5. saves the result to `data/processed/bank_with_dates.csv`, used by the later notebooks.

Source of the data: Moro, Cortez and Rita (2014), *A Data-Driven Approach to Predict the Success of Bank Telemarketing*, Decision Support Systems 62:22–31. UCI Machine Learning Repository, https://doi.org/10.24432/C5K306.

In [ ]:
import sys
from pathlib import Path

# Make the shared module in src/ importable from this notebook
sys.path.append(str(Path.cwd().parent))

In [ ]:
import pandas as pd

from src.data_utils import (load_raw, add_dates, assign_period, PERIODS,
                            LEAKAGE_COLUMNS, PROCESSED_FILE)

## 1. Read the raw file

In [ ]:
df = load_raw()
print(df.shape)
df.head()

The file should contain 41,188 calls and 21 columns (20 inputs and the target `y`).

In [ ]:
assert df.shape == (41188, 21), "Unexpected size: is this bank-additional-full.csv?"
df.dtypes

## 2. The target and missing values

In [ ]:
df["y"].value_counts(normalize=True).round(3)

Only about 11% of the clients subscribe. The classes are unbalanced, which is why the project uses ROC-AUC and precision-recall AUC instead of accuracy.

The file has no empty cells, but some categorical columns use the label `"unknown"` for a missing value. We count them here; they are kept as their own category.

In [ ]:
print("Empty cells:", df.isna().sum().sum())

unknown_share = (df == "unknown").mean()
unknown_share[unknown_share > 0].sort_values(ascending=False).round(3)

## 3. Rebuild the year of each call

The file only gives the month and the day of the week. The UCI documentation states that the rows are **ordered by date, from May 2008 to November 2010**. So a new year starts every time the month goes back (for example from `dec` to `mar`). The function `add_dates` in `src/data_utils.py` applies this rule.

In [ ]:
df = add_dates(df)
df["year"].value_counts().sort_index()

**Check.** If the reconstruction is right, the years must run from 2008 to 2010, and the 3-month Euribor rate in each month must follow its known history: close to 5% in mid-2008, falling sharply from the end of 2008, and below 1% from late 2009.

In [ ]:
assert df["year"].min() == 2008 and df["year"].max() == 2010
assert df["year_month"].is_monotonic_increasing, "Rows are not in date order"

monthly = df.groupby("year_month").agg(
    calls=("y", "size"),
    yes_rate=("y", lambda s: (s == "yes").mean()),
    euribor_min=("euribor3m", "min"),
    euribor_max=("euribor3m", "max"),
)
monthly.round(3)

The Euribor path matches its history, so the reconstructed years can be trusted. Some months have very few calls (for example December 2008), and there are no calls in January, February or September 2008.

## 4. Assign the time periods

The periods are defined once, in `src/data_utils.py`:

| Period | Months | Role |
|---|---|---|
| train | May–Aug 2008 | fit the models |
| validation | Oct–Dec 2008 | tune the models, still before the rate collapse |
| test_1 | Mar–May 2009 | just after the collapse |
| test_2 | Jun 2009–Nov 2010 | the most distant period |

In [ ]:
df = assign_period(df)
assert (df["period"] != "unused").all(), "Some calls fall outside every period"

order = list(PERIODS)
summary = (df.groupby("period")
             .agg(calls=("y", "size"),
                  yes_rate=("y", lambda s: (s == "yes").mean()),
                  euribor_mean=("euribor3m", "mean"))
             .loc[order])
summary.round(3)

## 5. Remove the leakage column and save

`duration` (the length of the call) is known only after the call has ended, so it cannot be used to decide whom to call. It is removed here, before any modelling, so that no later notebook can use it by mistake.

In [ ]:
df = df.drop(columns=LEAKAGE_COLUMNS)

PROCESSED_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_FILE, index=False)
print("Saved", df.shape, "to", PROCESSED_FILE.relative_to(PROCESSED_FILE.parents[2]))